# GloVe, TF-IDF weighted mean, concatenated + 2-layer MLP

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Project constraint:** static, pretrained word embeddings that are never trained, and a
classifier with 2 layers and a ReLU non-linearity. Only the classifier is trained.

**Approach of this notebook.** Same pipeline as `../glove_concatenate_bigger_mlp.ipynb`, with these changes:

- the sentence vector is the **TF-IDF weighted mean** of the GloVe word vectors instead of the plain mean, so frequent words like "the", "was" and "he" count less
- the classifier is reduced to the 2 layers the project asks for: `Linear(1200, 256) → ReLU → Linear(256, 2)` since this is a project constrainted
- dropout in the classifier (`Dropout → Linear → ReLU → Dropout → Linear`), 0.1 by default against overfitting, tunable in the config. Dropout has no weights, so it stays 2 layers

### What the earlier static runs taught me

- **Averaging the four sentences cannot work.** The mean of O1, O2, H1, H2 does not change when H1 and H2 swap, so the loss stays at ln 2. Concatenating them in a fixed order fixes that.
- **Validation and test disagree.** A word-overlap rule without any model reaches 63.5% on validation and 51.9% on test. The models learn this shortcut. Test is very likely the adversarially filtered ART dev set.

In [1]:
%pip install -q datasets gensim lightning nltk python-dotenv scikit-learn wandb


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))  # notebooks/project_1 -> shared pipeline package (this notebook is one folder deeper)

import lightning as L
import torch
import torch.nn as nn

from pipeline import tracking
from pipeline.data import DATASET, TEXT_FIELDS, load_splits
from pipeline.features.static_embeddings import MeanEmbedder, TfidfMeanEmbedder, build_features, combine_concat, load_vectors
from pipeline.features.tfidf import fit_vectorizer, idf_weights
from pipeline.loaders import make_loader
from pipeline.preprocessing import tokenize_split
from pipeline.preprocessing import nltk_alnum_lower as preprocessing  # swap the preprocessing variant here
from pipeline.training import evaluate_best, fit, training_summary, weights_dir

In [3]:
tracking.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
config = {
    "dataset": DATASET,
    **preprocessing.CONFIG,  # tokenizer, lowercase, remove_punctuation, remove_stopwords, normalization, truncation
    "embedding_dim": 300,  # dimensions per word vector, glove-wiki-gigaword comes with 50, 100, 200 or 300
    "embedding_trainable": False,  # static: vectors are only looked up, never updated
    "sentence_representation": "weighted_mean",
    "sentence_weighting": "tfidf",  # "tfidf" or "none" (plain mean, for the ablation)
    "tfidf_fit_on": "train sentences (O1, O2, H1, H2)",
    "example_representation": "concat",  # O1, O2, H1, H2
    "classifier": "mlp",  # dropout -> linear -> relu -> dropout -> linear
    "hidden_dim": 256,  # the one hidden layer: linear(4 * embedding_dim, hidden_dim) -> relu -> linear(hidden_dim, 2)
    "dropout": 0.1,  # share of MLP inputs zeroed during training, minimal default against overfitting, tune here (0.0 = off)
    "learning_rate": 1e-3,
    "weight_decay": 0.0,  # AdamW: every step shrinks all weights a little towards 0, 0.0 = off (plain Adam), e.g. 1e-4 to 1e-2
    "gradient_clip_val": 0.0,  # max length (L2 norm) of all gradients together per step, longer ones are scaled down, 0.0 = off, e.g. 1.0
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 5,  # validation checks without improvement of loss/valid before stopping
    "seed": 42,
}
config["embedding"] = f"glove-wiki-gigaword-{config['embedding_dim']}"  # gensim model id, e.g. "glove-wiki-gigaword-300"
# the shapes in the code comments of this notebook are for the defaults above (300 dims per word, hidden_dim 256)

# one name per variant, so the ablation does not overwrite the checkpoint of the main run
RUN_NAME = {"tfidf": "glove-tfidf-mlp", "none": "glove-mean-mlp"}[config["sentence_weighting"]]

run = tracking.init_run(RUN_NAME, config)

In [5]:
train, valid, test = load_splits()

train: 16801, valid: 1000, test: 1532


### Preprocessing

In [6]:
train_tokens = tokenize_split(train, preprocessing.preprocess)
valid_tokens = tokenize_split(valid, preprocessing.preprocess)
test_tokens = tokenize_split(test, preprocessing.preprocess)

Map:   0%|          | 0/16801 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1532 [00:00<?, ? examples/s]

### Static word embeddings (frozen)

GloVe is loaded once and only used as a lookup table. The word vectors are turned into numpy
features before training starts, so no gradient ever reaches them. The only trainable weights
in this notebook are the two linear layers of the MLP below.

In [7]:
wv = load_vectors(config["embedding"])

vocabulary: 400000 words, 300 dimensions


### TF-IDF weighting

In [8]:
# documents for the idf: every single training sentence (O1, O2, H1, H2), already tokenized
vectorizer = fit_vectorizer(train_tokens)
idf, idf_unseen = idf_weights(vectorizer, train_tokens)  # dict word -> idf, idf for words never seen in training

vocabulary: 16189 terms
tf-idf fitted on 67204 training sentences, vocabulary: 16189 terms
idf range: 2.06 – 11.42, words never seen in train: 12.12


In [9]:
# 1. The 15 words with the lowest idf: they appear in the most training sentences, so they count the least
words_lowest_first = sorted(idf, key=idf.get)  # all vocabulary words, sorted by their idf value
lowest_words = words_lowest_first[:15]

parts = []
for word in lowest_words:
    parts.append(f"{word} {idf[word]:.2f}")  # e.g. "the 2.06"
print("lowest idf:", ", ".join(parts))
print()

# 2. How one sentence changes: share of every word in the plain mean vs. in the tf-idf weighted mean
sentence = train_tokens[0]["hypothesis_1"]  # ['he', 'was', 'never', 'lost']

weights = []
for word in sentence:
    weights.append(idf.get(word, idf_unseen))  # a word never seen in training gets the highest idf
total = sum(weights)

print(f"{'token':<15}{'plain mean':>12}{'tf-idf':>10}")  # <15 = left-aligned in 15 characters, >12 = right-aligned in 12
for word, weight in zip(sentence, weights):
    plain_share = 1 / len(sentence)  # plain mean: every word counts the same
    tfidf_share = weight / total     # tf-idf: a word counts by its idf, all shares add up to 100%
    print(f"{word:<15}{plain_share:>12.1%}{tfidf_share:>10.1%}")

lowest idf: the 2.06, to 2.20, a 2.24, was 2.47, and 2.66, her 3.07, his 3.17, he 3.20, i 3.28, in 3.33, she 3.39, for 3.43, of 3.49, had 3.50, it 3.52

token            plain mean    tf-idf
he                    25.0%     19.3%
was                   25.0%     14.9%
never                 25.0%     30.7%
lost                  25.0%     35.1%


### Sentence and example vectors

In [10]:
if config["sentence_weighting"] == "tfidf":
    embed = TfidfMeanEmbedder(wv, idf, idf_unseen)  # rare words count more, fixed weights: not learned
else:
    embed = MeanEmbedder(wv)  # plain mean, for the ablation

In [11]:
# per example: 4 sentence vectors (300,) glued in order O1, O2, H1, H2 -> one row (1200,)
# X_train (16801, 1200), X_valid (1000, 1200), X_test (1532, 1200), y (examples,): 0 = hypothesis_1, 1 = hypothesis_2

X_train, y_train = build_features(train_tokens, embed, combine_concat)
X_valid, y_valid = build_features(valid_tokens, embed, combine_concat)
X_test, y_test = build_features(test_tokens, embed, combine_concat)

print(f"input: {X_train.shape[1]} dimensions, {X_train.dtype}")
print(f"OOV: {embed.stats['oov']} of {embed.stats['tokens']} tokens ({embed.stats['oov'] / embed.stats['tokens']:.2%}), "
      f"empty sentences: {embed.stats['empty_sentences']}, in GloVe but not in the train idf: {embed.stats['tfidf_unseen']}")

input: 1200 dimensions, float32
OOV: 1144 of 670395 tokens (0.17%), empty sentences: 7, in GloVe but not in the train idf: 1417


### DataLoaders - Takes care of Training Batching and Loading

In [12]:
L.seed_everything(config["seed"])

warnings.filterwarnings("ignore", ".*does not have many workers.*")

# one batch: x (64, 1200), y (64,), the last batch of a split can be smaller
train_loader = make_loader(X_train, y_train, config["batch_size"], shuffle=True)
valid_loader = make_loader(X_valid, y_valid, config["batch_size"], shuffle=False)
test_loader = make_loader(X_test, y_test, config["batch_size"], shuffle=False)

Seed set to 42


### LightningModule Model (Derived from libraries.pdf using coding agents in combination with Lightning Documentation)

In [13]:
class MLP(L.LightningModule):

    def __init__(self, input_dim, hidden_dim, output_dim, dropout, learning_rate, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        # 2 layers as the project asks: linear -> relu -> linear. These are the only trainable weights
        # dropout zeroes a random share of the values in training only (eval/test use all of them), it has no weights
        self.layers = nn.Sequential(
            nn.Dropout(dropout),  # (batch, 1200) -> (batch, 1200), random features zeroed
            # weights (256, 1200) + bias (256,): (batch, 1200) -> (batch, 256)
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),  # (batch, 256) -> (batch, 256), random hidden units zeroed
            # weights (2, 256) + bias (2,): (batch, 256) -> logits (batch, 2)
            nn.Linear(hidden_dim, output_dim),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def reset_parameters(self):
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                layer.reset_parameters()

    def forward(self, x):
        # x (batch, 1200) -> dropout -> linear -> relu -> dropout -> linear -> logits (batch, 2)
        return self.layers(x)

    def configure_optimizers(self):
        # AdamW = Adam with decoupled weight decay: each step multiplies every weight by (1 - learning_rate * weight_decay).
        # Adam's own weight_decay is added to the gradient instead, where the adaptive scaling weakens it. 0.0 = plain Adam
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=self.hparams.weight_decay)

    def shared_step(self, batch, split):
        x, y = batch
        # x (batch, 1200) -> logits (batch, 2), y (batch,)
        logits = self(x)
        # logits (batch, 2), y (batch,) -> loss: one number, mean over the batch
        loss = self.loss_fn(logits, y)
        # argmax over the 2 logits (batch, 2) -> answer (batch,), == y (batch,) -> mean: share of correct answers
        accuracy = (logits.argmax(dim=1) == y).float().mean()
        self.log(f"loss/{split}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"accuracy/{split}", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    # @todo check that against ML libraries
    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, "valid")

    def test_step(self, batch, batch_idx):
        return self.shared_step(batch, "test")

In [14]:
model = MLP(
    input_dim=len(TEXT_FIELDS) * wv.vector_size,
    hidden_dim=config["hidden_dim"],
    output_dim=2,
    dropout=config["dropout"],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
model.reset_parameters()
# weights (256, 1200) + bias (256,) + weights (2, 256) + bias (2,) = 307,970
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

trainer = fit(
    model, train_loader, valid_loader, weights_dir(RUN_NAME),
    max_epochs=config["max_epochs"], patience=config["patience"], gradient_clip_val=config["gradient_clip_val"],
)

stopped_epoch = trainer.early_stopping_callback.stopped_epoch  # read right after fit, evaluate_best resets it to 0
print(f"early stopping in epoch {stopped_epoch} (0-based), best checkpoint: {Path(trainer.checkpoint_callback.best_model_path).name}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_tfidf_mlp/weights exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layers  │ Sequential       │  307 K │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 307 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 307 K                                                                                                
Total estimated model params size (MB): 1.232                                                                      
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

Metric loss/valid improved. New best score: 0.677
Metric loss/valid improved by 0.011 >= min_delta = 0.0. New best score: 0.666
Metric loss/valid improved by 0.007 >= min_delta = 0.0. New best score: 0.660
IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Monitored metric loss/valid did not improve in the last 5 records. Best score: 0.652. Signaling Trainer to stop.


early stopping in epoch 8 (0-based), best checkpoint: best-epoch=3-v2.ckpt


### Best Checkpoint + Upload Artifacts to Weights and Biases

In [15]:
evaluate_best(trainer, model, valid_loader, test_loader)

run.summary.update({
    **training_summary(trainer, y_test),  # accuracy/majority, best/loss_valid, best/checkpoint
    "early_stopping/stopped_epoch": stopped_epoch,  # read right after fit, see above
    "model/trainable_parameters": trainable_parameters,
    **embed.summary(),  # oov/..., for tf-idf also tfidf/vocabulary_size, tfidf/unseen_tokens
})

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_tfidf_mlp/weights/best-epoch=3-v2.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_tfidf_mlp/weights/best-epoch=3-v2.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      accuracy/valid       │    0.5960000157356262     │
│        loss/valid         │    0.6521248817443848     │
└───────────────────────────┴───────────────────────────┘

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_tfidf_mlp/weights/best-epoch=3-v2.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_tfidf_mlp/weights/best-epoch=3-v2.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       accuracy/test       │    0.5208877325057983     │
│         loss/test         │    0.7559483647346497     │
└───────────────────────────┴───────────────────────────┘

In [ ]:
tracking.log_best_checkpoint(run, RUN_NAME, trainer.checkpoint_callback)
run.finish()